# Muscle Ultrasound Segmentation → PA / FL / MT (v2 – thin-fascicle fixes)
**Why fascicle segmentation was poor, and what v2 does about it**
1. **Thin lines were destroyed by resizing** (nearest-neighbour shrink to 512×640 lost most 1–3 px fascicle pixels) → canvas now follows the real image size (up to 1024 px), masks are resized with area-averaging + low threshold.
2. **Target too thin to learn** → fascicle masks are dilated by 2 px for training (the geometry step does not need 1 px precision).
3. **Plain Dice on 1–3 px lines is brutally harsh** → added a boundary-tolerant F1 (±3 px) next to Dice/IoU.
4. **Loss** → weighted BCE + Focal-Tversky + **clDice** (rewards connected, continuous lines).
5. **Model** → full-resolution skip stream into the head, so fine detail isn't lost in the decoder.
6. **Training on 384×384 random crops** centred on fascicles (80 %) → more informative steps, bigger batch (12), 40 epochs.
7. **Hysteresis thresholding** at inference to repair broken fascicles; threshold tuned on val; flip TTA.
8. **Inspection cell** after training shows image / GT / probability / prediction on unseen val scans.

## 1. Setup

In [ ]:
import os, re, sys, glob, random, math, time, json, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

cv2.setNumThreads(0)  # avoid CPU oversubscription with DataLoader workers

class CFG:
    MAX_SIDE = 1024                 # canvas = median image size (aspect kept, padded to /32), long side capped here
    PX_PER_MM = 8.0                 # !! VERIFY against the challenge docs / scale bar of the test images !!
    BATCH, VAL_BATCH = 12, 4        # training uses 384x384 crops; validation/test use the full canvas
    CROP, WIN = 384, 448            # random crop size / window that is augmented before the centre-crop
    EPOCHS_FASC, EPOCHS_APO = 40, 30
    FASC_DILATE = 2                 # thicken thin fascicle masks by this many px (3x3 ellipse) for training
    APO_DILATE = 0
    TOL_PX = 3                      # tolerance (px) for the boundary-tolerant F1 metric
    CL_ITERS = 5                    # soft-skeleton iterations for the clDice loss term
    HYST_LO = 0.6                   # hysteresis: low threshold = THR * HYST_LO (fascicle post-processing)
    LR = 3e-4
    PATIENCE = 10                   # early stopping on val Dice
    PRETRAINED = True               # ImageNet ResNet34 encoder
    TTA = True                      # horizontal-flip test-time augmentation
    SEED = 42
    MEAN, STD = 0.449, 0.226

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

def seed_everything(s=CFG.SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything()
torch.backends.cudnn.benchmark = True


## 2. Data (Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_data_dir = Path('/content/drive/MyDrive/ml data')
if not drive_data_dir.exists():
    alt = Path('/content/drive/MyDrive/mldata')
    if alt.exists(): drive_data_dir = alt
    else: raise FileNotFoundError(f"Dataset folder not found. Folders in MyDrive: {[p.name for p in Path('/content/drive/MyDrive').iterdir() if p.is_dir()]}")

target_dir = Path('/content/dataset')
zip_files = list(drive_data_dir.glob('*.zip'))
if zip_files:
    if not target_dir.exists() or not any(target_dir.iterdir()):      # don't re-extract after a runtime reconnect
        print(f'[*] Unzipping {zip_files[0].name} ...')
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_files[0]) as z: z.extractall(target_dir)
else:
    if target_dir.is_symlink(): target_dir.unlink()
    elif target_dir.exists(): shutil.rmtree(target_dir)
    os.symlink(str(drive_data_dir), str(target_dir))

def find_dir(name):
    for root in [target_dir, drive_data_dir]:
        if (root / name).is_dir(): return root / name
        hits = [p for p in root.rglob(name) if p.is_dir()]
        if hits: return hits[0]
    return None

FASC_IMG_DIR, FASC_MASK_DIR = find_dir('fasc_imgs'), find_dir('fasc_masks')
APO_IMG_DIR,  APO_MASK_DIR  = find_dir('apo_imgs'),  find_dir('apo_masks')
TEST_IMG_DIR = find_dir('test_imgs')
for k, v in dict(FASC_IMG_DIR=FASC_IMG_DIR, FASC_MASK_DIR=FASC_MASK_DIR, APO_IMG_DIR=APO_IMG_DIR, APO_MASK_DIR=APO_MASK_DIR, TEST_IMG_DIR=TEST_IMG_DIR).items():
    print(f'{k:14s}: {v}')
assert all([FASC_IMG_DIR, FASC_MASK_DIR, APO_IMG_DIR, APO_MASK_DIR, TEST_IMG_DIR]), 'A dataset folder was not found'


## 3. Audit pairs, sizes, stroke width

In [ ]:
EXTS = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}

def get_paired_files(img_dir, mask_dir):
    """Match by file STEM (so img.png <-> img.tif still pairs) and report anything unmatched."""
    imgs  = sorted(p for p in img_dir.iterdir()  if p.suffix.lower() in EXTS and not p.name.startswith('.'))
    masks = {p.stem: p for p in mask_dir.iterdir() if p.suffix.lower() in EXTS and not p.name.startswith('.')}
    pairs = [(i, masks[i.stem]) for i in imgs if i.stem in masks]
    print(f'  images={len(imgs)} masks={len(masks)} matched={len(pairs)} (unmatched images: {len(imgs)-len(pairs)})')
    return pairs

def binarize_mask(arr):
    # BUG FIX: the old code used `> 127`, which silently yields an ALL-ZERO mask if masks are stored as 0/1
    return (arr > 0) if arr.max() <= 1 else (arr > 127)

print('Fascicle:');    fasc_pairs = get_paired_files(FASC_IMG_DIR, FASC_MASK_DIR)
print('Aponeurosis:'); apo_pairs  = get_paired_files(APO_IMG_DIR,  APO_MASK_DIR)

# --- data sanity audit: sizes, mask values, foreground fraction ---
def mask_width(m):
    """Approximate stroke width in px: 2*area/perimeter of the foreground (thin lines -> 1-3 px)."""
    m = m.astype(np.uint8)
    if m.sum() == 0: return np.nan
    cnts, _ = cv2.findContours(m, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    per = sum(cv2.arcLength(c, True) for c in cnts)
    return 2.0 * float(m.sum()) / max(per, 1.0)

def audit(pairs, n=60):
    rows = []
    for ip, mp in random.sample(pairs, min(n, len(pairs))):
        im, mk = np.array(Image.open(ip)), np.array(Image.open(mp))
        rows.append(dict(img_hw=im.shape[:2], mask_hw=mk.shape[:2], img_dtype=str(im.dtype), mask_max=int(mk.max()), fg=float(binarize_mask(mk.astype(np.int64)).mean()), width=mask_width(binarize_mask(mk.astype(np.int64)))))
    d = pd.DataFrame(rows)
    print('  image sizes   :', d.img_hw.value_counts().to_dict())
    print('  size mismatch :', int((d.img_hw != d.mask_hw).sum()), '| dtypes:', d.img_dtype.unique(), '| mask max values:', sorted(d.mask_max.unique()))
    print(f'  foreground fraction: mean={d.fg.mean():.4f}  min={d.fg.min():.4f}  max={d.fg.max():.4f}  (empty masks: {(d.fg==0).sum()})')
    print(f'  median stroke width at native resolution: {d.width.median():.1f} px  (<=3 px = thin lines; this is why dilation + hi-res are used)')
print('Fascicle audit:');    audit(fasc_pairs)
print('Aponeurosis audit:'); audit(apo_pairs)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for r, (name, pairs) in enumerate([('Fascicle', fasc_pairs), ('Aponeurosis', apo_pairs)]):
    ip, mp = random.choice(pairs)
    im = np.array(Image.open(ip).convert('L')); mk = binarize_mask(np.array(Image.open(mp).convert('L')))
    if mk.shape != im.shape: mk = cv2.resize(mk.astype(np.uint8), im.shape[::-1], interpolation=cv2.INTER_NEAREST) > 0
    ov = np.stack([im]*3, -1); ov[mk] = (0.5*ov[mk] + 0.5*np.array([0, 255, 120])).astype(np.uint8)
    for a, x, t in zip(axes[r], [im, mk, ov], [f'{name}: {ip.name}', 'mask', 'overlay']):
        a.imshow(x, cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()


## 4. Group-wise split

In [ ]:
def make_group_id(stem):
    """Try to recover subject/sweep id from the filename by stripping the trailing frame number."""
    return re.sub(r'[_\-\s]*\d+$', '', stem)

def create_subject_splits(pairs, train_pct=0.70, val_pct=0.15, seed=CFG.SEED, frames_per_sweep=50):
    df = pd.DataFrame({'image_path': [str(i) for i, _ in pairs], 'mask_path': [str(m) for _, m in pairs],
                       'stem': [i.stem for i, _ in pairs]})
    df['group'] = df.stem.map(make_group_id)
    n_groups = df.group.nunique()
    if n_groups < 10 or n_groups > 0.9 * len(df):
        # filenames carry no usable subject id -> contiguous blocks (neighbouring frames never straddle splits)
        df['group'] = [f'blk_{i // frames_per_sweep:04d}' for i in range(len(df))]
        print(f'  [!] no usable subject id in filenames -> using blocks of {frames_per_sweep} consecutive frames '
              f'({df.group.nunique()} groups). Check this matches your real subject structure to avoid leakage.')
    else:
        print(f'  grouped by filename prefix: {n_groups} groups')
    groups = sorted(df.group.unique())
    np.random.RandomState(seed).shuffle(groups)
    n_tr, n_va = int(round(len(groups) * train_pct)), int(round(len(groups) * val_pct))
    split_of = {g: ('train' if i < n_tr else 'val' if i < n_tr + n_va else 'test') for i, g in enumerate(groups)}
    df['split'] = df.group.map(split_of)
    return df

print('Fascicle:');    fasc_df = create_subject_splits(fasc_pairs)
print('Aponeurosis:'); apo_df  = create_subject_splits(apo_pairs)
for n, d in [('Fascicle', fasc_df), ('Aponeurosis', apo_df)]:
    print(f'{n:12s}', d.split.value_counts().to_dict())


## 5. Canvas, cache, crops, augmentation

In [ ]:
def ceil32(v): return int(math.ceil(v / 32) * 32)

def task_canvas(pairs, n=40):
    """Canvas = median image size, aspect preserved, long side capped at MAX_SIDE, both sides multiples of 32.
    (The old fixed 512x640 canvas shrank thin fascicle lines until they were barely visible.)"""
    sz = [Image.open(ip).size for ip, _ in random.sample(pairs, min(n, len(pairs)))]    # (w, h)
    w, h = int(np.median([s[0] for s in sz])), int(np.median([s[1] for s in sz]))
    sc = min(1.0, CFG.MAX_SIDE / max(h, w))
    return ceil32(h * sc), ceil32(w * sc)

CANVAS = {'fascicle': task_canvas(fasc_pairs), 'aponeurosis': task_canvas(apo_pairs)}
print('Canvas (H, W):', CANVAS)

def letterbox(arr, H, W, interp):
    """Resize keeping aspect ratio, pad bottom/right with zeros (plain stretching distorts fascicle angles)."""
    h, w = arr.shape[:2]
    s = min(H / h, W / w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    out = np.zeros((H, W), arr.dtype)
    out[:nh, :nw] = cv2.resize(arr, (nw, nh), interpolation=interp)
    return out, (h, w, nh, nw)

def prep_mask(mk, H, W, dilate):
    """Binary mask -> canvas. Down-scaling uses area averaging + a LOW threshold so 1-2 px lines survive
    (nearest-neighbour resizing drops/shifts thin lines). Optional dilation makes the target learnable."""
    h, w = mk.shape
    down = min(H / h, W / w) < 1
    m, _ = letterbox(mk.astype(np.float32), H, W, cv2.INTER_AREA if down else cv2.INTER_LINEAR)
    m = (m > (0.15 if down else 0.5)).astype(np.uint8)
    if dilate > 0:
        m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=dilate)
    return m

def build_cache(df, name, dilate):
    H, W = CANVAS[name]
    cache = Path(f'/content/cache_{name}_{H}x{W}_d{dilate}.npz')
    if cache.exists():
        z = np.load(cache)
        if len(z['X']) == len(df): print(f'[cache] loaded {cache.name}'); return z['X'], z['Y']
    print(f'[cache] building {name}: ~{2 * len(df) * H * W / 1e9:.1f} GB in RAM')
    X = np.zeros((len(df), H, W), np.uint8); Y = np.zeros_like(X)
    for i, r in enumerate(df.itertuples()):
        im = np.array(Image.open(r.image_path).convert('L'))
        mk = binarize_mask(np.array(Image.open(r.mask_path).convert('L')).astype(np.int64)).astype(np.uint8)
        if mk.shape != im.shape: mk = cv2.resize(mk, im.shape[::-1], interpolation=cv2.INTER_NEAREST)
        X[i], _ = letterbox(im, H, W, cv2.INTER_AREA if min(H / im.shape[0], W / im.shape[1]) < 1 else cv2.INTER_LINEAR)
        Y[i] = prep_mask(mk, H, W, dilate)
    np.savez(cache, X=X, Y=Y); print(f'[cache] saved {cache.name}  X={X.shape}')
    return X, Y

def sample_window(msk, wh, ww, pos_prob=0.8):
    """Random window; with prob pos_prob it is centred near a foreground pixel so crops are not mostly empty."""
    H, W = msk.shape
    if random.random() < pos_prob:
        ys, xs = np.nonzero(msk[::4, ::4])                      # sub-sampled for speed
        if len(ys):
            k = random.randrange(len(ys)); cy, cx = ys[k] * 4, xs[k] * 4
            y0 = int(np.clip(cy - wh // 2 + random.randint(-wh // 4, wh // 4), 0, H - wh))
            x0 = int(np.clip(cx - ww // 2 + random.randint(-ww // 4, ww // 4), 0, W - ww))
            return y0, x0
    return random.randint(0, H - wh), random.randint(0, W - ww)

def augment(img, msk):
    H, W = img.shape
    if random.random() < 0.5:
        img, msk = img[:, ::-1], msk[:, ::-1]
    if random.random() < 0.8:                                   # small affine
        M = cv2.getRotationMatrix2D((W / 2, H / 2), random.uniform(-8, 8), random.uniform(0.9, 1.1))
        M[:, 2] += (random.uniform(-0.05, 0.05) * W, random.uniform(-0.05, 0.05) * H)
        img = cv2.warpAffine(np.ascontiguousarray(img), M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        msk = cv2.warpAffine(np.ascontiguousarray(msk), M, (W, H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT_101)
    img = img.astype(np.float32) / 255.0
    if random.random() < 0.7:                                   # gamma + contrast/brightness
        img = np.clip(img, 0, 1) ** random.uniform(0.7, 1.4)
        img = img * random.uniform(0.85, 1.15) + random.uniform(-0.08, 0.08)
    if random.random() < 0.5:                                   # speckle-like noise
        img = img * (1 + np.random.randn(H, W).astype(np.float32) * random.uniform(0.0, 0.08))
        img = img + np.random.randn(H, W).astype(np.float32) * random.uniform(0.0, 0.02)
    if random.random() < 0.2:
        img = cv2.GaussianBlur(img, (0, 0), random.uniform(0.5, 1.2))
    return np.clip(img, 0, 1), np.ascontiguousarray(msk)

class SegDataset(Dataset):
    """train: random crops (CROP x CROP) from the full canvas;  val/test: the full canvas."""
    def __init__(self, X, Y, idx, train):
        self.X, self.Y, self.idx, self.train = X, Y, idx, train      # shares X/Y (no per-split copies -> half the RAM)
        H, W = X.shape[1:]
        self.wh, self.ww = min(CFG.WIN, H), min(CFG.WIN, W)
        self.ch, self.cw = min(CFG.CROP, H), min(CFG.CROP, W)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        img, msk = self.X[self.idx[i]], self.Y[self.idx[i]]
        if self.train:
            y0, x0 = sample_window(msk, self.wh, self.ww)
            img, msk = augment(img[y0:y0 + self.wh, x0:x0 + self.ww], msk[y0:y0 + self.wh, x0:x0 + self.ww])
            oy, ox = (self.wh - self.ch) // 2, (self.ww - self.cw) // 2
            img, msk = img[oy:oy + self.ch, ox:ox + self.cw], msk[oy:oy + self.ch, ox:ox + self.cw]
        else:
            img = img.astype(np.float32) / 255.0
        img = (img - CFG.MEAN) / CFG.STD
        return torch.from_numpy(np.ascontiguousarray(img)).unsqueeze(0).float(), torch.from_numpy(np.ascontiguousarray(msk).astype(np.float32)).unsqueeze(0)

def make_datasets(df, name, dilate):
    X, Y = build_cache(df, name, dilate)
    idx = {s: np.where(df.split.values == s)[0] for s in ['train', 'val', 'test']}
    ds = {s: SegDataset(X, Y, ix, train=(s == 'train')) for s, ix in idx.items()}
    ds['train_Y_mean'] = float(Y[idx['train']].mean())
    return ds

def make_loaders(ds):
    kw = dict(num_workers=2, pin_memory=True, persistent_workers=True)
    return (DataLoader(ds['train'], CFG.BATCH, shuffle=True, drop_last=True, **kw),
            DataLoader(ds['val'], CFG.VAL_BATCH, shuffle=False, num_workers=2, pin_memory=True))

fasc_ds, apo_ds = make_datasets(fasc_df, 'fascicle', CFG.FASC_DILATE), make_datasets(apo_df, 'aponeurosis', CFG.APO_DILATE)
fasc_train_loader, fasc_val_loader = make_loaders(fasc_ds)
apo_train_loader,  apo_val_loader  = make_loaders(apo_ds)
print('Batches/epoch  fascicle:', len(fasc_train_loader), '| aponeurosis:', len(apo_train_loader))

def fg_frac(ds): return ds['train_Y_mean']
print(f"Foreground fraction (after dilation)  fascicle={fg_frac(fasc_ds):.4f}  aponeurosis={fg_frac(apo_ds):.4f}")


## 6. Model, loss, metrics

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                 nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class AttentionGate(nn.Module):
    """Additive attention gate (Attention U-Net) on the skip connection."""
    def __init__(self, g_ch, x_ch, inter):
        super().__init__()
        self.wg = nn.Sequential(nn.Conv2d(g_ch, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.wx = nn.Sequential(nn.Conv2d(x_ch, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.psi = nn.Sequential(nn.Conv2d(inter, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        g = F.interpolate(self.wg(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(F.relu(g + self.wx(x), inplace=True))

class DecBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, attention=True):
        super().__init__()
        self.att = AttentionGate(in_ch, skip_ch, max(skip_ch // 2, 16)) if attention else None
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        if self.att is not None: skip = self.att(x, skip)
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], 1))

class ResNetUNet(nn.Module):
    """U-Net with an ImageNet-pretrained ResNet34 encoder + attention-gated skips. 1-channel input."""
    def __init__(self, pretrained=True, attention=True, num_classes=1):
        super().__init__()
        w = None
        if pretrained:
            try: w = models.ResNet34_Weights.IMAGENET1K_V1
            except Exception: w = None
        try:
            r = models.resnet34(weights=w)
        except Exception as e:
            print('[!] could not download pretrained weights, training encoder from scratch:', e)
            r = models.resnet34(weights=None)
        conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        conv1.weight.data = r.conv1.weight.data.sum(1, keepdim=True)         # RGB -> gray
        self.stem = nn.Sequential(conv1, r.bn1, r.relu)                       # /2   64
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4   # /4 64, /8 128, /16 256, /32 512
        self.d4 = DecBlock(512, 256, 256, attention)
        self.d3 = DecBlock(256, 128, 128, attention)
        self.d2 = DecBlock(128, 64, 64, attention)
        self.d1 = DecBlock(64, 64, 64, attention)
        self.hr = ConvBlock(1, 16)                                            # full-resolution stream: keeps thin-line detail
        self.head = nn.Sequential(ConvBlock(64 + 16, 32), nn.Conv2d(32, num_classes, 1))
    def forward(self, x):
        H, W = x.shape[2:]
        s0 = self.stem(x); s1 = self.l1(self.pool(s0)); s2 = self.l2(s1); s3 = self.l3(s2); s4 = self.l4(s3)
        d = self.d4(s4, s3); d = self.d3(d, s2); d = self.d2(d, s1); d = self.d1(d, s0)
        d = F.interpolate(d, size=(H, W), mode='bilinear', align_corners=False)
        return self.head(torch.cat([d, self.hr(x)], 1))

def build_model():
    return ResNetUNet(pretrained=CFG.PRETRAINED, attention=True).to(device).to(memory_format=torch.channels_last)

# ---- loss: weighted BCE + Focal-Tversky (recall-oriented) + optional clDice (keeps thin lines connected) ----
def _erode(x):
    return torch.min(-F.max_pool2d(-x, (3, 1), 1, (1, 0)), -F.max_pool2d(-x, (1, 3), 1, (0, 1)))
def _dilate(x): return F.max_pool2d(x, 3, 1, 1)
def soft_skeleton(x, iters):
    skel = F.relu(x - _dilate(_erode(x)))
    for _ in range(iters):
        x = _erode(x)
        delta = F.relu(x - _dilate(_erode(x)))
        skel = skel + F.relu(delta - skel * delta)
    return skel

class ComboLoss(nn.Module):
    def __init__(self, fg_fraction, alpha=0.3, beta=0.7, gamma=0.75, w_cl=0.0, cl_iters=5):
        super().__init__()
        self.pw = float(min(math.sqrt((1 - fg_fraction) / max(fg_fraction, 1e-6)), 8.0))
        self.alpha, self.beta, self.gamma, self.w_cl, self.cl_iters = alpha, beta, gamma, w_cl, cl_iters
    def forward(self, logits, t):
        logits = logits.float()
        bce = F.binary_cross_entropy_with_logits(logits, t, pos_weight=torch.tensor(self.pw, device=t.device))
        p = torch.sigmoid(logits)
        tp, fp, fn = (p * t).sum((1, 2, 3)), (p * (1 - t)).sum((1, 2, 3)), ((1 - p) * t).sum((1, 2, 3))
        ti = (tp + 1) / (tp + self.alpha * fp + self.beta * fn + 1)
        loss = bce + ((1 - ti) ** self.gamma).mean()
        if self.w_cl > 0:
            sp, st = soft_skeleton(p, self.cl_iters), soft_skeleton(t, self.cl_iters)
            prec = ((sp * t).sum((1, 2, 3)) + 1) / (sp.sum((1, 2, 3)) + 1)
            sens = ((st * p).sum((1, 2, 3)) + 1) / (st.sum((1, 2, 3)) + 1)
            loss = loss + self.w_cl * (1 - 2 * prec * sens / (prec + sens + 1e-6)).mean()
        return loss

# ---- metrics (per-image Dice / IoU, then averaged) ----
def dice_iou(prob, target, thr):
    p = (prob > thr).float()
    inter = (p * target).sum((1, 2, 3)); ps, ts = p.sum((1, 2, 3)), target.sum((1, 2, 3))
    dice = torch.where((ps + ts) == 0, torch.ones_like(inter), (2 * inter) / (ps + ts + 1e-6))
    iou  = torch.where((ps + ts) == 0, torch.ones_like(inter), inter / (ps + ts - inter + 1e-6))
    return dice, iou

def tol_f1(prob, target, thr, tol):
    """Boundary-tolerant F1: a predicted pixel counts if it is within `tol` px of the target (and vice versa).
    Plain Dice is brutally harsh on 1-3 px wide lines (a 1 px shift = 0 overlap), so this is the fairer score."""
    p = (prob > thr).float(); k = 2 * tol + 1
    t_d, p_d = F.max_pool2d(target, k, 1, tol), F.max_pool2d(p, k, 1, tol)
    ps, ts = p.sum((1, 2, 3)), target.sum((1, 2, 3))
    prec = (p * t_d).sum((1, 2, 3)) / (ps + 1e-6); rec = (target * p_d).sum((1, 2, 3)) / (ts + 1e-6)
    f1 = 2 * prec * rec / (prec + rec + 1e-6)
    return torch.where((ps + ts) == 0, torch.ones_like(f1), f1)

@torch.no_grad()
def predict_prob(model, x, tta=False):
    """x: normalised (B,1,H,W). Returns probabilities (B,1,H,W); optional hflip TTA."""
    x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
    with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
        p = torch.sigmoid(model(x).float())
        if tta: p = 0.5 * (p + torch.sigmoid(model(x.flip(-1)).float()).flip(-1))
    return p


## 7. Train (fascicle uses clDice)

In [ ]:
@torch.no_grad()
def evaluate(model, loader, thr=0.5, tta=False):
    model.eval(); D, I, T = [], [], []
    for x, y in loader:
        p = predict_prob(model, x, tta); y = y.to(device)
        d, i = dice_iou(p, y, thr); D.append(d); I.append(i); T.append(tol_f1(p, y, thr, CFG.TOL_PX))
    return torch.cat(D).mean().item(), torch.cat(I).mean().item(), torch.cat(T).mean().item()

def train_segmentor(task, train_loader, val_loader, fg_fraction, epochs, w_cl=0.0, lr=CFG.LR):
    print(f'\n{"="*64}\n TRAINING {task.upper()}  ({epochs} epochs, early-stop patience {CFG.PATIENCE})\n{"="*64}')
    seed_everything()
    model = build_model()
    criterion = ComboLoss(fg_fraction, w_cl=w_cl, cl_iters=CFG.CL_ITERS)
    # lower LR for the pretrained encoder, higher for the freshly initialised decoder
    enc = [p for n, p in model.named_parameters() if n.split('.')[0] in ('stem', 'l1', 'l2', 'l3', 'l4')]
    dec = [p for n, p in model.named_parameters() if n.split('.')[0] not in ('stem', 'l1', 'l2', 'l3', 'l4')]
    opt = torch.optim.AdamW([{'params': enc, 'lr': lr * 0.3}, {'params': dec, 'lr': lr}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr * 0.3, lr], total_steps=epochs * len(train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
    ckpt, best, bad, hist = f'{task}_best.pth', -1.0, 0, []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); tl = 0.0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last); y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(x)
            loss = criterion(logits, y)              # loss in fp32, outside autocast
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item()
        tl /= len(train_loader)
        vd, vi, vf = evaluate(model, val_loader, thr=0.5)
        hist.append((tl, vd, vi, vf))
        improved = vd > best
        if improved: best, bad = vd, 0; torch.save(model.state_dict(), ckpt)
        else: bad += 1
        print(f'Ep {ep:02d}/{epochs} | train loss {tl:.4f} | val Dice {vd:.4f} IoU {vi:.4f} tolF1 {vf:.4f} | {(time.time()-t0)/60:.1f} min{"  <- best" if improved else ""}')
        if bad >= CFG.PATIENCE: print(f'[early stop] no val-Dice gain for {CFG.PATIENCE} epochs'); break
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print(f'[+] {task}: best val Dice {best:.4f} -> {ckpt}')
    h = np.array(hist); fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h[:, 0]); ax[0].set_title(f'{task}: train loss'); ax[1].plot(h[:, 1], label='Dice'); ax[1].plot(h[:, 2], label='IoU'); ax[1].plot(h[:, 3], label=f'tolF1({CFG.TOL_PX}px)'); ax[1].legend(); ax[1].set_title('val')
    plt.show()
    return model

fasc_model = train_segmentor('fascicle',    fasc_train_loader, fasc_val_loader, fg_frac(fasc_ds), CFG.EPOCHS_FASC, w_cl=0.5)
apo_model  = train_segmentor('aponeurosis', apo_train_loader,  apo_val_loader,  fg_frac(apo_ds),  CFG.EPOCHS_APO)

# ---- look at what the fascicle model actually predicts on unseen VAL images (always inspect this!) ----
@torch.no_grad()
def show_val_predictions(model, ds, thr=0.5, n=3):
    fig, axes = plt.subplots(n, 4, figsize=(18, 4.2 * n))
    for r, i in enumerate(random.sample(range(len(ds['val'])), min(n, len(ds['val'])))):
        x, y = ds['val'][i]; p = predict_prob(model, x[None], CFG.TTA)[0, 0].cpu().numpy()
        for a, im, t in zip(axes[r], [x[0].numpy(), y[0].numpy(), p, p > thr], ['image', 'ground truth (dilated)', 'probability', f'pred > {thr}']):
            a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.show()
show_val_predictions(fasc_model, fasc_ds)


## 8. Threshold tuning + hold-out test score

In [ ]:
# Tune the binarisation threshold on VAL (the old notebook used 0.4 for val and 0.35 for test, never tuned),
# then report an honest, untouched hold-out score on the TEST split.
@torch.no_grad()
def tune_threshold(model, ds, thrs=np.arange(0.20, 0.81, 0.05)):
    loader = DataLoader(ds['val'], CFG.VAL_BATCH, shuffle=False)
    scores = {round(float(t), 2): [] for t in thrs}
    model.eval()
    for x, y in loader:
        p = predict_prob(model, x, CFG.TTA); y = y.to(device)
        for t in scores: scores[t].append(dice_iou(p, y, t)[0])
    scores = {t: torch.cat(v).mean().item() for t, v in scores.items()}
    best = max(scores, key=scores.get)
    return best, scores

THR = {}
for name, model, ds in [('fascicle', fasc_model, fasc_ds), ('aponeurosis', apo_model, apo_ds)]:
    THR[name], sc = tune_threshold(model, ds)
    test_loader = DataLoader(ds['test'], CFG.VAL_BATCH, shuffle=False)
    td, ti, tf = evaluate(model, test_loader, THR[name], CFG.TTA)
    print(f'{name:12s} best thr={THR[name]:.2f} (val Dice {sc[THR[name]]:.4f}) | HOLD-OUT test Dice {td:.4f}  IoU {ti:.4f}  tolF1 {tf:.4f}')


## 9. Clinical geometry (PA, FL, MT)

In [ ]:
# ---------- Clinical geometry (runs on ORIGINAL-resolution masks) ----------
def _components(mask, min_area):
    n, lab, stats, cents = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    return [(i, stats[i, cv2.CC_STAT_AREA], cents[i]) for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area], lab

def fit_aponeuroses(apo_mask, min_area=300):
    """Return (superficial, deep) as dicts with line y = a*x + b, or None if two sheets are not found."""
    H, W = apo_mask.shape
    comps, lab = _components(apo_mask, min_area)
    sheets = []
    for i, area, _ in comps:
        ys, xs = np.nonzero(lab == i)
        if np.ptp(xs) < 0.15 * W:            # too short horizontally to be an aponeurosis
            continue
        cols = np.unique(xs)                  # per-column median y = robust centre-line
        cy = np.array([np.median(ys[xs == c]) for c in cols])
        a, b = np.polyfit(cols, cy, 1)
        sheets.append(dict(a=a, b=b, area=area, x0=cols.min(), x1=cols.max(), my=cy.mean()))
    if len(sheets) < 2:
        return None
    sheets.sort(key=lambda s: -s['area'])
    first = sheets[0]
    # second sheet = largest component that is clearly above/below the first
    others = [s for s in sheets[1:] if abs(s['my'] - first['my']) > 0.08 * H]
    if not others:
        return None
    second = others[0]
    sup, deep = sorted([first, second], key=lambda s: s['my'])
    return sup, deep

def muscle_thickness_px(sup, deep):
    x0, x1 = max(sup['x0'], deep['x0']), min(sup['x1'], deep['x1'])
    if x1 - x0 < 20:                           # no overlap -> use the union
        x0, x1 = min(sup['x0'], deep['x0']), max(sup['x1'], deep['x1'])
    xs = np.arange(x0, x1 + 1)
    gap = (deep['a'] * xs + deep['b']) - (sup['a'] * xs + sup['b'])
    mean_slope = 0.5 * (sup['a'] + deep['a'])
    return float(np.median(gap) * math.cos(math.atan(mean_slope)))   # perpendicular distance

def fascicle_stats(fasc_mask, sup, deep, min_area=80):
    """Per-fascicle PA (relative to deep aponeurosis) and FL (fascicle extrapolated to both aponeuroses)."""
    comps, lab = _components(fasc_mask, min_area)
    apo_ang = math.degrees(math.atan(deep['a']))
    pas, fls, wts = [], [], []
    for i, area, _ in comps:
        ys, xs = np.nonzero(lab == i)
        pts = np.column_stack([xs, ys]).astype(np.float32)
        vx, vy, cx, cy = [float(v) for v in cv2.fitLine(pts, cv2.DIST_L2, 0, 0.01, 0.01).ravel()]
        ang = math.degrees(math.atan2(vy, vx))
        ang = (ang + 90) % 180 - 90                       # fold to (-90, 90]
        pa = abs(ang - apo_ang)
        pa = min(pa, 180 - pa)
        if not (3 <= pa <= 60):                           # not a plausible fascicle
            continue
        # intersect the fascicle line with both aponeurosis lines
        pts_int = []
        for ln in (sup, deep):
            den = vy - ln['a'] * vx
            if abs(den) < 1e-6:
                break
            t = (ln['a'] * cx + ln['b'] - cy) / den
            pts_int.append((cx + t * vx, cy + t * vy))
        else:
            fl = math.hypot(pts_int[0][0] - pts_int[1][0], pts_int[0][1] - pts_int[1][1])
            fls.append(fl)
            pas.append(pa); wts.append(len(xs))
            continue
        pas.append(pa); fls.append(np.nan); wts.append(len(xs))
    if not pas:
        return None
    pas, fls, wts = np.array(pas), np.array(fls), np.array(wts, dtype=float)
    order = np.argsort(pas); cw = np.cumsum(wts[order]) / wts.sum()
    pa = float(pas[order][np.searchsorted(cw, 0.5)])      # length-weighted median
    fl = float(np.nanmedian(fls)) if np.isfinite(fls).any() else np.nan
    return pa, fl

def calculate_clinical_parameters(fasc_mask, apo_mask, px_per_mm):
    """Returns dict(pa_deg, fl_mm, mt_mm) with NaN where a measurement failed (filled later, not hard-coded)."""
    out = dict(pa_deg=np.nan, fl_mm=np.nan, mt_mm=np.nan)
    sheets = fit_aponeuroses(apo_mask)
    if sheets is None:
        return out
    sup, deep = sheets
    mt_px = muscle_thickness_px(sup, deep)
    out['mt_mm'] = mt_px / px_per_mm
    fs = fascicle_stats(fasc_mask, sup, deep)
    if fs is not None:
        pa, fl_px = fs
        out['pa_deg'] = pa
        out['fl_mm'] = (fl_px / px_per_mm) if np.isfinite(fl_px) else out['mt_mm'] / math.sin(math.radians(pa))
    return out


## 10. Test inference → submission.csv

In [ ]:
from google.colab import files

test_files = sorted(p for p in TEST_IMG_DIR.iterdir() if p.suffix.lower() in EXTS and not p.name.startswith('.'))
print(f'[+] {len(test_files)} test images')

def hysteresis(prob, hi, lo):
    """Keep low-threshold regions that touch a confident (high-threshold) pixel -> fewer broken fascicles."""
    n, lab = cv2.connectedComponents((prob > lo).astype(np.uint8), connectivity=8)
    keep = np.unique(lab[prob > hi]); keep = keep[keep > 0]
    return np.isin(lab, keep).astype(np.uint8)

@torch.no_grad()
def segment_full_res(model, gray, thr, canvas, hyst=False):
    """Letterbox -> predict (+TTA) -> crop padding -> resize prob back to ORIGINAL size -> threshold."""
    H, W = canvas
    lb, (h, w, nh, nw) = letterbox(gray, H, W, cv2.INTER_AREA if min(H / gray.shape[0], W / gray.shape[1]) < 1 else cv2.INTER_LINEAR)
    x = torch.from_numpy(((lb.astype(np.float32) / 255 - CFG.MEAN) / CFG.STD))[None, None]
    p = predict_prob(model, x, CFG.TTA)[0, 0, :nh, :nw].cpu().numpy()
    p = cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR)
    m = hysteresis(p, thr, thr * CFG.HYST_LO) if hyst else (p > thr).astype(np.uint8)
    return m, p

records = []
for img_p in test_files:
    gray = np.array(Image.open(img_p).convert('L'))
    m_f, _ = segment_full_res(fasc_model, gray, THR['fascicle'], CANVAS['fascicle'], hyst=True)
    m_a, _ = segment_full_res(apo_model,  gray, THR['aponeurosis'], CANVAS['aponeurosis'])
    r = calculate_clinical_parameters(m_f, m_a, CFG.PX_PER_MM)
    records.append(dict(image_id=img_p.name, **r))
sub_df = pd.DataFrame(records)

# Failed measurements: fill with the MEDIAN of successful predictions (not a made-up constant), and say how many.
for c in ['pa_deg', 'fl_mm', 'mt_mm']:
    n_bad = int(sub_df[c].isna().sum())
    print(f'{c}: {n_bad}/{len(sub_df)} measurements failed -> filled with median {sub_df[c].median():.2f}')
    sub_df[c] = sub_df[c].fillna(sub_df[c].median())
sub_df[['pa_deg', 'fl_mm', 'mt_mm']] = sub_df[['pa_deg', 'fl_mm', 'mt_mm']].round(3)
sub_df.to_csv('submission.csv', index=False)
print(sub_df.head(10)); print(sub_df[['pa_deg', 'fl_mm', 'mt_mm']].describe())

# Compare against sample_submission if the dataset provides one (column names / id format must match exactly)
ss = list(drive_data_dir.rglob('sample_submission*.csv')) + list(target_dir.rglob('sample_submission*.csv'))
if ss:
    s = pd.read_csv(ss[0]); print('\n[check] sample_submission columns:', list(s.columns), '| ours:', list(sub_df.columns))
    if list(s.columns) != list(sub_df.columns): print('[!] COLUMN NAMES DIFFER from sample_submission - rename before uploading')

try:
    for f in ['submission.csv', 'fascicle_best.pth', 'aponeurosis_best.pth']: shutil.copy(f, drive_data_dir / f)
    print('[+] copies saved to Drive')
except Exception as e: print('[!] Drive copy failed:', e)
files.download('submission.csv')


## 11. Visual check

In [ ]:
samples = random.sample(test_files, min(4, len(test_files)))
fig, axes = plt.subplots(2, 2, figsize=(15, 10)); axes = axes.ravel()
for ax, img_p in zip(axes, samples):
    gray = np.array(Image.open(img_p).convert('L'))
    m_f, _ = segment_full_res(fasc_model, gray, THR['fascicle'], CANVAS['fascicle'], hyst=True); m_a, _ = segment_full_res(apo_model, gray, THR['aponeurosis'], CANVAS['aponeurosis'])
    rgb = np.stack([gray] * 3, -1)
    rgb[m_f > 0] = (0.5 * rgb[m_f > 0] + 0.5 * np.array([0, 255, 120])).astype(np.uint8)
    rgb[m_a > 0] = (0.5 * rgb[m_a > 0] + 0.5 * np.array([255, 40, 180])).astype(np.uint8)
    ax.imshow(rgb); ax.axis('off')
    sh = fit_aponeuroses(m_a)
    if sh:                                              # draw the fitted aponeurosis lines used for MT
        xs = np.array([0, gray.shape[1] - 1])
        for ln in sh: ax.plot(xs, ln['a'] * xs + ln['b'], 'c--', lw=1.5)
    row = sub_df[sub_df.image_id == img_p.name].iloc[0]
    ax.set_title(f"{img_p.name}\nPA {row.pa_deg:.1f}° | FL {row.fl_mm:.1f} mm | MT {row.mt_mm:.1f} mm", fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()
